# 03 — Aggregation
Loads `data/processed/variants_filtered.parquet` from notebook 02 and produces window-density and population AF summaries via `src/aggregate.py`.

In [1]:
import sys, os
from pathlib import Path

import pandas as pd

ROOT = Path(os.getcwd()).parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from aggregate import compute_window_density, melt_population_af

In [2]:
# ── Configure ──────────────────────────────────────────────────────────────
IN_PATH = ROOT / 'data' / 'processed' / 'variants_filtered.parquet'
OUT_DIR = ROOT / 'data' / 'processed'

WINDOW_SIZE = 100_000   # base pairs

In [3]:
# ── Load ───────────────────────────────────────────────────────────────────
variants = pd.read_parquet(IN_PATH)
print(f'Loaded {len(variants):,} variants')
variants.head()

Loaded 37,089 variants


,chrom,pos,ref,alt,qual,filter_status,af_global,af_afr,af_eur,af_eas,af_amr,af_sas,ac,an,variant_type,source_file
0,chrY,2781758,C,T,None,PASS,0.002024,0.000000,0.000000,0.032258,0.006849,0.000000,3,1482,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
1,chrY,2781761,CAAAAAAAA,C,None,PASS,0.002469,0.013158,0.000000,0.000000,0.000000,0.000000,3,1215,INDEL,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
2,chrY,2782660,G,A,None,PASS,0.001632,0.001269,0.002425,0.000000,0.002979,0.000000,55,33697,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
3,chrY,2782986,C,T,None,PASS,0.001306,0.000000,0.000000,0.000000,0.000000,0.030028,43,32920,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz
4,chrY,2783081,G,A,None,PASS,0.004827,0.003770,0.009193,0.000000,0.000294,0.000000,152,31491,SNP,gnomad.genomes.v4.1.sites.chrY.vcf.bgz


In [4]:
# ── Window density ─────────────────────────────────────────────────────────
window_df = compute_window_density(variants, window_size=WINDOW_SIZE)
print(f'{len(window_df):,} windows')
window_df.head()

213 windows


,window,variant_count,mean_af,snp_count,indel_count
0,2700000,40,0.017088,30,10
1,2800000,200,0.021573,140,60
2,2900000,271,0.016695,158,113
3,3000000,175,0.015916,134,41
4,3100000,181,0.012943,136,45


In [5]:
# ── Population AF (long format) ────────────────────────────────────────────
pop_df = melt_population_af(variants)
print(f'{len(pop_df):,} variant-population rows')
pop_df.head()

185,134 variant-population rows


,pos,variant_type,population,allele_frequency
0,2781758,SNP,African,0.000000
1,2781761,INDEL,African,0.013158
2,2782660,SNP,African,0.001269
3,2782986,SNP,African,0.000000
4,2783081,SNP,African,0.003770


In [6]:
# ── Save ───────────────────────────────────────────────────────────────────
window_df.to_parquet(OUT_DIR / 'window_density.parquet', index=False, engine='pyarrow')
pop_df.to_parquet(OUT_DIR / 'population_af.parquet', index=False, engine='pyarrow')
print(f'Saved → {OUT_DIR / "window_density.parquet"}')
print(f'Saved → {OUT_DIR / "population_af.parquet"}')

Saved → /mnt/c/IISER/SEM_8/ds_practice/genomic-variation-landscape-and-population-comparison-main/genomic-variation-landscape-and-population-comparison-main/data/processed/window_density.parquet
Saved → /mnt/c/IISER/SEM_8/ds_practice/genomic-variation-landscape-and-population-comparison-main/genomic-variation-landscape-and-population-comparison-main/data/processed/population_af.parquet
